# PXR Challenge — EDA & Data Curation
Load raw train/test, standardise SMILES, filter quality outliers, visualise chemical space, apply applicability-domain filter, and save curated training set.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Draw
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

ROOT = Path('.')
sys.path.insert(0, str(ROOT))
from utils import (
    COL_NAME, COL_PECSO, COL_SE, COL_SMILES,
    ecfp4_fp, mol_to_inchikey, remove_salts, smiles_to_mol,
    DATA_DIR,
)

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

## 1. Load raw data

In [ ]:
train = pd.read_csv(DATA_DIR / 'train_raw.csv')
test  = pd.read_csv(DATA_DIR / 'test_raw.csv')
print(f'Train: {train.shape}  Test: {test.shape}')
train.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train[COL_PECSO].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_xlabel('pEC50'); axes[0].set_title('pEC50 distribution (train)')
train[COL_SE].hist(bins=50, ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_xlabel('pEC50 std.error'); axes[1].set_title('Measurement uncertainty')
plt.tight_layout(); plt.savefig(ROOT / 'results' / 'pEC50_distribution.png', dpi=150); plt.show()

## 2. SMILES standardisation

In [ ]:
def standardise_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    mols, canonical, flags = [], [], []
    for smi in df[COL_SMILES]:
        mol = smiles_to_mol(str(smi))
        if mol is None:
            mols.append(None); canonical.append(None); flags.append('invalid_smiles')
        else:
            mol = remove_salts(mol)
            mols.append(mol)
            canonical.append(Chem.MolToSmiles(mol))
            flags.append('ok')
    df['mol'] = mols
    df['canonical_smiles'] = canonical
    df['curation_flag'] = flags
    return df

train = standardise_df(train)
test  = standardise_df(test)

print('Train invalid SMILES:', (train['curation_flag'] != 'ok').sum())
print('Test  invalid SMILES:', (test['curation_flag']  != 'ok').sum())

## 3. Quality filters

In [ ]:
# --- InChIKey deduplication (keep row with lowest std.error) ---------------
train['inchikey'] = train['mol'].apply(lambda m: mol_to_inchikey(m) if m is not None else None)
test['inchikey']  = test['mol'].apply(lambda m: mol_to_inchikey(m) if m is not None else None)

dupes_before = train.shape[0]
has_key = train['inchikey'].notna()
train_keyed  = train[has_key].sort_values(COL_SE).drop_duplicates(subset='inchikey', keep='first')
train_no_key = train[~has_key]
train = pd.concat([train_keyed, train_no_key], ignore_index=True)
print(f'Removed {dupes_before - train.shape[0]} duplicate InChIKeys')

test_keys = set(test['inchikey'].dropna())
print(f'Train molecules also in test (exact match): {train["inchikey"].isin(test_keys).sum()}')

In [ ]:
# --- Reliability filter: high std.error → unreliable fit --------------------
SE_THRESHOLD = 0.5   # pEC50 units; adjust after inspecting histogram above
mask_se = train[COL_SE] > SE_THRESHOLD
print(f'Removed {mask_se.sum()} rows with pEC50_std.error > {SE_THRESHOLD}')

# --- Extreme pEC50 filter ---------------------------------------------------
PECSO_MIN, PECSO_MAX = 1.0, 8.5
mask_extreme = (train[COL_PECSO] < PECSO_MIN) | (train[COL_PECSO] > PECSO_MAX)
print(f'Removed {mask_extreme.sum()} rows with pEC50 outside [{PECSO_MIN}, {PECSO_MAX}]')

# --- Invalid mol -----------------------------------------------------------
mask_invalid = train['curation_flag'] != 'ok'
print(f'Removed {mask_invalid.sum()} rows with invalid SMILES')

# Apply all filters
remove_mask = mask_se | mask_extreme | mask_invalid
train_curated = train[~remove_mask].copy()
print(f'\nTrain after curation: {len(train_curated)} / {len(train)} ({100*(1-len(train_curated)/len(train)):.1f}% removed)')

## 4. Chemical space visualisation (PCA + t-SNE)

In [ ]:
from rdkit import DataStructs
import warnings

def fps_to_array(mols, n_bits=2048):
    arr = np.zeros((len(mols), n_bits), dtype=np.uint8)
    for i, m in enumerate(mols):
        if m is not None:
            fp = ecfp4_fp(m, n_bits)
            DataStructs.ConvertToNumpyArray(fp, arr[i])
    return arr

train_mols_ok = train_curated['mol'].tolist()
test_mols_ok  = [m for m in test['mol'] if m is not None]

print(f'Computing fingerprints for {len(train_mols_ok)} train + {len(test_mols_ok)} test molecules...')
fp_train = fps_to_array(train_mols_ok)
fp_test  = fps_to_array(test_mols_ok)

all_fps = np.vstack([fp_train, fp_test])
labels = np.array(['train'] * len(fp_train) + ['test'] * len(fp_test))
print('Done.')

In [ ]:
# PCA
pca = PCA(n_components=2, random_state=42)
coords_pca = pca.fit_transform(all_fps)
print(f'PCA variance explained: {pca.explained_variance_ratio_[:2].sum():.1%}')

df_pca = pd.DataFrame({
    'PC1': coords_pca[:, 0], 'PC2': coords_pca[:, 1],
    'split': labels,
    'pEC50': list(train_curated[COL_PECSO].values) + [np.nan] * len(fp_test),
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Colour by split
for split, colour in [('train', 'steelblue'), ('test', 'tomato')]:
    sub = df_pca[df_pca['split'] == split]
    axes[0].scatter(sub['PC1'], sub['PC2'], c=colour, alpha=0.4, s=8, label=split)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].set_title('PCA — Train vs Test')
axes[0].legend()

# Colour by pEC50 (train only)
tr = df_pca[df_pca['split'] == 'train']
sc = axes[1].scatter(tr['PC1'], tr['PC2'], c=tr['pEC50'], cmap='viridis', alpha=0.5, s=8)
plt.colorbar(sc, ax=axes[1], label='pEC50')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('PCA — pEC50 activity landscape')

plt.tight_layout()
plt.savefig(ROOT / 'results' / 'chemical_space_pca.png', dpi=150)
plt.show()

In [ ]:
# t-SNE (subsample if large)
MAX_TSNE = 3000
if len(all_fps) > MAX_TSNE:
    rng = np.random.default_rng(0)
    idx = rng.choice(len(all_fps), MAX_TSNE, replace=False)
    all_fps_sub = all_fps[idx]
    labels_sub  = labels[idx]
else:
    all_fps_sub, labels_sub = all_fps, labels

print(f't-SNE on {len(all_fps_sub)} molecules...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
coords_tsne = tsne.fit_transform(all_fps_sub)
print('Done.')

fig, ax = plt.subplots(figsize=(8, 6))
for split, colour in [('train', 'steelblue'), ('test', 'tomato')]:
    mask = labels_sub == split
    ax.scatter(coords_tsne[mask, 0], coords_tsne[mask, 1],
               c=colour, alpha=0.4, s=8, label=split)
ax.set_title('t-SNE — Train vs Test')
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'chemical_space_tsne.png', dpi=150)
plt.show()

## 5. Applicability domain (AD) filtering
Remove training molecules with very low maximum Tanimoto similarity to any test compound. These are likely out-of-domain and may hurt generalisation.

In [ ]:
from rdkit import DataStructs

test_fps_rdkit = [
    ecfp4_fp(m) for m in test_mols_ok
]
train_fps_rdkit = [
    ecfp4_fp(m) for m in train_mols_ok
]

print(f'Computing max Tanimoto similarity to test for {len(train_fps_rdkit)} train molecules...')
max_sim = np.array([
    max(DataStructs.BulkTanimotoSimilarity(fp, test_fps_rdkit))
    for fp in train_fps_rdkit
])
train_curated = train_curated.copy()
train_curated['max_sim_to_test'] = max_sim
print(f'Max Tanimoto — mean={max_sim.mean():.3f}  median={np.median(max_sim):.3f}  min={max_sim.min():.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(max_sim, bins=60, color='steelblue', edgecolor='white')
ax.axvline(0.15, color='red', linestyle='--', label='default threshold (0.15)')
ax.set_xlabel('Max Tanimoto similarity to test set')
ax.set_title('Applicability Domain — Train→Test similarity')
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'ad_similarity_histogram.png', dpi=150)
plt.show()

for thresh in [0.05, 0.10, 0.15, 0.20, 0.25]:
    n_remove = (max_sim < thresh).sum()
    print(f'  threshold={thresh:.2f}  →  {n_remove} removed  ({100*n_remove/len(max_sim):.1f}%)')

In [ ]:
# Set threshold here after visual inspection of the histogram above
AD_THRESHOLD = 0.15   # <-- adjust as needed

ad_mask = train_curated['max_sim_to_test'] < AD_THRESHOLD
print(f'Removing {ad_mask.sum()} molecules below AD threshold {AD_THRESHOLD}')
train_final_pre_chembl = train_curated[~ad_mask].reset_index(drop=True)
print(f'Train after AD filter: {len(train_final_pre_chembl)}')

## 6. Save curated datasets

In [ ]:
# Keep only essential columns for downstream scripts
keep_cols_train = [COL_NAME, 'canonical_smiles', COL_PECSO, COL_SE,
                   'inchikey', 'max_sim_to_test', 'source']
keep_cols_test  = [COL_NAME, 'canonical_smiles', 'inchikey']

out_train = train_final_pre_chembl[keep_cols_train].copy()
out_train.rename(columns={'canonical_smiles': 'SMILES'}, inplace=True)
out_train.to_csv(DATA_DIR / 'train_curated.csv', index=False)

out_test = test[keep_cols_test].copy()
out_test.rename(columns={'canonical_smiles': 'SMILES'}, inplace=True)
out_test.to_csv(DATA_DIR / 'test_curated.csv', index=False)

print(f'Saved train_curated.csv  : {len(out_train)} rows')
print(f'Saved test_curated.csv   : {len(out_test)} rows')
print(f'\npEC50 range in curated train: [{out_train[COL_PECSO].min():.2f}, {out_train[COL_PECSO].max():.2f}]')
out_train.describe()

## 7. Summary of removed compounds

In [ ]:
print('=== Curation Summary ===')
print(f'  Raw train         : {len(train)}')
print(f'  After dedup       : {len(train[train["curation_flag"]=="ok"])}')
print(f'  After SE filter   : -{mask_se.sum()}')
print(f'  After extreme pEC50: -{mask_extreme.sum()}')
print(f'  After AD filter   : -{ad_mask.sum()}')
print(f'  Final curated     : {len(out_train)}')